# Day 8 - LLM foundations & API integration practical tasks
***

These tasks cover calling OpenAI API's and prompting.

## Explore how different model parameters and different prompts influence the response.
#### Here are some guidelines and advice you should consider:
* You can add as many cells as you need for each task.
* Use Markdown cells for your conclusions.
* For tasks that do not specify a model, you may choose any of the two available models: `gpt-5.6-luna` / `gpt-5.4-mini`.
* Within each comparison task, change only the model, prompt, or parameter specified in the instructions. Keep everything else - including the model, prompt, and other settings - the same across calls so you can compare the results fairly. For example, when testing different `temperature` values, use the same model, prompt, and output token limit (`max_output_tokens` for the Responses API or `max_completion_tokens` for the Chat Completions API).
* You can use both - [Responses API and Chat Completions API](https://medium.com/@sanathgoutham/openai-responses-api-vs-chat-completions-what-actually-changed-eeee86aff216) - to complete these tasks, but some model parameters are not supported by Responses API.
* LLMs understand [Markdown](https://www.markdownguide.org/cheat-sheet/) formatting in prompts. For example, wrapping text in double asterisks `(**important text**)` marks it as **bold** and can help signal emphasis.
* To copy text from a Markdown cell, open it in edit mode by clicking the pencil icon or double-clicking the cell.
* Optionally, paste API responses into a Markdown cell for better readability.
* Be creative and experiment.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ.pop("OPENAI_API_KEY", None)
load_dotenv()

True

In [2]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in the W2D3 .env file before running this cell.")

client = OpenAI(base_url="https://ds-ai-internship.openai.azure.com/openai/v1/",
                api_key=OPENAI_API_KEY)

In [3]:
import time
from IPython.display import Markdown, display

# one helper for all tasks (Chat Completions, because it supports n and the penalties)
def ask(prompt, system=None, model="gpt-5.4-mini", **settings):
    messages = [{"role": "user", "content": prompt}]
    if system:
        messages.insert(0, {"role": "system", "content": system})
    return client.chat.completions.create(model=model, messages=messages, **settings)

def show(reply):
    for i, choice in enumerate(reply.choices, 1):
        print(f"[{i}]", choice.message.content)
        print()

**Note:** LLM answers are not deterministic. Even with the same prompt and settings, every run gives slightly different text (except some repeats at `temperature=0`).
The conclusions below are based on the outputs saved in this notebook. If the notebook is re-run, the answers will change and the conclusions may not match them exactly.

#### 1. Write a Python script to call the OpenAI API and generate a response to the prompt below.

Prompt: `"What is artificial intelligence?"`

Use model `gpt-5.6-luna`.

In [4]:
reply = ask("What is artificial intelligence?", model="gpt-5.6-luna")
show(reply)

[1] Artificial intelligence (AI) is the field of creating computer systems that can perform tasks that typically require human intelligence, such as:

- Understanding language
- Recognizing images and speech
- Learning from data
- Solving problems
- Making predictions or decisions
- Generating text, images, music, or code

AI systems use algorithms and data to identify patterns and produce outputs. Some AI is designed for specific tasks, like recommending videos or detecting fraud, while more general AI would be capable of handling a wide range of intellectual tasks.



#### 2. Call the OpenAI API with different models

Prompt `"Explain the concept of machine learning to a 10-year-old."`

`model`: `gpt-5.6-luna` and `gpt-5.4-mini`.

Compare the outputs from each model. 

You can evaluate the differences in:
* Clarity of explanation.
* Simplicity of language (suitable for a 10-year-old).
* Creativity in the response.
* Response time.

In [5]:
kid_prompt = "Explain the concept of machine learning to a 10-year-old."

start = time.time()
luna = ask(kid_prompt, model="gpt-5.6-luna")
print(f"gpt-5.6-luna: {time.time() - start:.1f} s, {luna.usage.completion_tokens} output tokens\n")
show(luna)

gpt-5.6-luna: 2.2 s, 153 output tokens

[1] Machine learning is a way of teaching computers to make guesses or decisions by showing them lots of examples.

Imagine you want a computer to recognize cats. Instead of writing every rule about cats—like “has whiskers” or “has pointy ears”—you show it thousands of pictures labeled **cat** or **not cat**. The computer looks for patterns, such as shapes, colors, and features.

After practicing, it can look at a new picture and make a guess: “I think this is a cat!”

It’s called **machine learning** because the computer improves by learning from examples, a bit like how people get better at something with practice. However, it doesn’t think or understand exactly like a person—it finds patterns in information.



In [6]:
start = time.time()
mini = ask(kid_prompt, model="gpt-5.4-mini")
print(f"gpt-5.4-mini: {time.time() - start:.1f} s, {mini.usage.completion_tokens} output tokens\n")
show(mini)

gpt-5.4-mini: 1.7 s, 197 output tokens

[1] Machine learning is a way to teach computers to learn from examples, kind of like how you learn from practice.

For example:

- If you show a computer lots of pictures of cats and dogs,
- it can start to figure out which ones are cats and which ones are dogs by itself.

It’s like giving the computer a big pile of homework with answers, so it can spot patterns and get better at making guesses.

### Simple idea:
- **People usually tell computers exactly what to do**
- **Machine learning lets computers learn what to do by looking at examples**

### Everyday example:
Imagine learning to recognize apples and oranges:
- At first, you might mix them up
- But after seeing many of them, you notice:
  - apples are often round and smooth
  - oranges are bumpy and orange

A computer can do the same thing with machine learning.

### In short:
Machine learning is when computers learn from experience, just like people do.



**Comparison**

| | gpt-5.6-luna | gpt-5.4-mini |
|---|---|---|
| Response time | 2.2 s | 1.7 s |
| Output tokens | 153 | 197 |
| Clarity | One example (recognizing cats), explained step by step | Several examples (cats/dogs, apples/oranges) with headings |
| Simplicity | Simple words, short flowing text | Simple words too, "homework with answers" fits a kid well |
| Creativity | Quite plain, but adds that the computer doesn't really "think" like a person | More creative, more analogies, feels like a school worksheet |

Both answers are fine for a 10-year-old. Luna is shorter and more focused, mini is more structured and playful but a bit longer.
The speed difference comes from one run only, so it is not very reliable.

#### 3. Call the OpenAI API with different `temperature` values

Prompt: `"Write a two sentence story about a robot learning to cook."`

`temperature` values: `0`, `0.7` and `1.5`. Generate three responses at each temperature.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare the outputs.

Tip: With Chat Completions API you can use parameter `n=3` to generate three different responses to your prompt with one call.

In [7]:
story = "Write a two sentence story about a robot learning to cook."

show(ask(story, temperature=0, n=3, reasoning_effort="none"))

[1] The robot carefully studied recipes, measuring flour and stirring soup with precise, mechanical grace, but it accidentally mistook salt for sugar and served a very confused cake. After a moment of blinking lights and whirring gears, it adjusted its sensors, tried again, and proudly presented a delicious meal that made everyone cheer.

[2] The robot carefully studied recipes, its metal fingers clinking as it chopped vegetables and stirred a bubbling pot with precise, mechanical grace. When it finally served dinner, the soup was a little too salty, but the robot glowed with pride because it had made its first meal all by itself.

[3] The robot carefully studied recipes, its metal fingers clinking as it chopped vegetables and stirred a bubbling pot with precise, mechanical grace. When it finally served dinner, the soup was a little too salty, but the robot glowed with pride because it had made its first meal all by itself.



In [8]:
show(ask(story, temperature=0.7, n=3, reasoning_effort="none"))

[1] The robot carefully studied the recipe, its metal fingers clumsily cracking eggs and stirring soup with surprising determination. By the time the kitchen filled with the warm smell of garlic and herbs, it had not only made dinner but also learned that cooking was a little like solving a delicious puzzle.

[2] The robot carefully studied recipes, whisked batter with precise mechanical arms, and proudly placed a golden loaf into the oven. When the kitchen filled with the warm smell of bread, it beeped with delight, realizing that cooking was less about perfect measurements and more about making something to share.

[3] The robot carefully studied recipes, its metal fingers clumsily chopping vegetables and stirring pots with precise little whirs. By the end of the evening, it served a surprisingly delicious soup and proudly announced that it had finally learned the secret ingredient: patience.



In [9]:
show(ask(story, temperature=1.5, n=3, reasoning_effort="none"))

[1] The robot watched carefully as Grandma stirred the soup, then copied her movements with cautious precision, only once dropping a spoon into the pot. By dinner time, it had served a perfectly salted stew and learned that the secret ingredient was patience.

[2] The robot stood in the kitchen, carefully studying a recipe and clumsily measuring flour with its metal fingers as steam fogged its bright blue eyes. By the time it served its first dish, a little lopsided but delicious, it realized that cooking was just another way to learn how to care.

[3] The robot carefully stirred the soup, consulting its recipe sensors as it learned that a pinch of salt could transform a bland broth into something magical. When its timer chimed, it proudly served the meal to its amazed creators, who realized they had built a chef with a stainless-steel heart.



**Observations**
- `temperature = 0`: almost no variety. Answers 2 and 3 are word-for-word the same, and all three start with "The robot carefully studied recipes".
  (It is not 100% deterministic, answer 1 is a bit different.)
- `temperature = 0.7`: all three stories are different (soup, bread, "the secret ingredient: patience"), but they still start the same way and have the same structure.
- `temperature = 1.5`: the most variety: different openings (Grandma, steam fogging the robot's eyes), different endings ("a chef with a stainless-steel heart").
  The text is still coherent, the model did not start writing nonsense.

Higher temperature makes the model pick less likely words more often, so the answers become more diverse and creative. Low temperature is better when we want the same answer every time.

#### 4. Call the OpenAI API with different `max_output_tokens`/`max_completion_tokens` values.

Prompt: `"Explain the concept of machine learning in simple terms."`

`max_output_tokens` (Response API call)/`max_completion_tokens` (Chat Completions API call) values: `50`, `100` and `200`.

Use `gpt-5.4-mini` for this task.

Compare the length and detail of the responses.

In [10]:
ml = "Explain the concept of machine learning in simple terms."

def limited(tokens):
    reply = ask(ml, max_completion_tokens=tokens)
    print(f"limit {tokens}: used {reply.usage.completion_tokens} tokens, finish_reason = {reply.choices[0].finish_reason}\n")
    show(reply)

limited(50)

limit 50: used 50 tokens, finish_reason = length

[1] Machine learning is a way of teaching computers to learn from examples instead of giving them every rule by hand.

### Simple idea
Imagine you want a computer to recognize cats in photos.  
Instead of writing thousands of rules like:
- “



In [11]:
limited(100)

limit 100: used 100 tokens, finish_reason = length

[1] Machine learning is a way for computers to **learn from examples** instead of being told every rule step by step.

### Simple idea
Imagine teaching a child to recognize cats:
- You show many pictures of cats and dogs.
- Over time, the child starts noticing patterns: ears, whiskers, shape, fur.
- Then the child can tell whether a new picture is a cat or a dog.

Machine learning works similarly:
- A computer is given lots of data
- It



In [12]:
limited(200)

limit 200: used 200 tokens, finish_reason = length

[1] Machine learning is a way for computers to **learn from data instead of being explicitly programmed for every rule**.

### Simple idea
Imagine teaching a computer to recognize cats in photos.

- In traditional programming, you would write lots of rules like:
  - look for pointy ears
  - look for whiskers
  - look for a certain face shape
- In machine learning, you give the computer **many examples** of cat and non-cat photos.
- The computer finds patterns on its own and learns how to tell the difference.

### In everyday terms
It’s a bit like how a child learns:
- You show examples
- The child notices patterns
- With practice, they get better at recognizing things

### Where it’s used
Machine learning helps with things like:
- recommendation systems on Netflix or YouTube
- spam email detection
- voice assistants like Siri or Alexa
- self-driving car features
- fraud detection in banking

### Short definition




**Observations**

| Limit | Tokens used | finish_reason | What we got |
|---|---|---|---|
| 50 | 50 | length | a one-sentence definition and the start of an example, cut in the middle |
| 100 | 100 | length | definition + "teaching a child" analogy, cut mid-sentence |
| 200 | 200 | length | definition, analogy, everyday terms, real-world uses, cut right before "Short definition" |

All three answers were cut off, the model did not "know" about the limit and did not shorten its answer, it just got stopped.
More tokens = more detail, but `max_completion_tokens` is a hard cut, not a way to ask for a short answer.
If we want a short complete answer, it is better to ask for it in the prompt (e.g. "in 3 sentences") and use the limit only as a safety net.

#### 5. Call the OpenAI API with different `top_p` values.

Prompt: `"Write a list of 5 completely random and unusual hobbies that no one has ever heard of."`

`top_p` values: `0.3`, `0.7`, and `1.0`.

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare response variation as you adjust `top_p`.

In [13]:
hobbies = "Write a list of 5 completely random and unusual hobbies that no one has ever heard of."

# n=3 so we can see how much the answers differ at each top_p
show(ask(hobbies, top_p=0.3, n=3, reasoning_effort="none"))

[1] Here are 5 completely random and unusual hobby ideas:

1. **Cloud Shadow Mapping**  
   Tracking and sketching the shapes of shadows cast by clouds at different times of day.

2. **Teacup Echo Listening**  
   Holding teacups in different places to hear how tiny echoes and vibrations change in each one.

3. **Button History Collecting**  
   Searching for old buttons and inventing fictional backstories for the people who might have worn them.

4. **Pebble Orchestra Conducting**  
   Arranging pebbles by size and tapping them in patterns like a miniature percussion ensemble.

5. **Invisible Maze Designing**  
   Drawing maze layouts meant only to be imagined, then “walking” through them mentally.

[2] Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow spotting** — tracking and sketching the shapes that clouds cast on the ground at different times of day.  
2. **Vintage key collecting by sound** — collecting old keys and identifying them by the way they clink to

In [14]:
show(ask(hobbies, top_p=0.7, n=3, reasoning_effort="none"))

[1] Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow tracing** — tracking the shapes and movement of shadows cast by clouds across different surfaces.
2. **Antique button cataloging** — collecting, sorting, and documenting old buttons by material, era, and pattern.
3. **Train station acoustics spotting** — visiting stations to notice and compare the unique echo and sound behavior of each one.
4. **Miniature door crafting** — making tiny decorative doors and placing them in unexpected places.
5. **Rain smell journaling** — recording and comparing the scent of rain in different locations, weather conditions, and seasons.

If you want, I can make them even weirder or turn them into fake “official” hobby names.

[2] Here are 5 completely random and unusual hobby ideas:

1. **Cloud shadow spotting** — tracking the shapes and movement of shadows cast by clouds at different times of day.  
2. **Vintage button cataloging** — collecting, sorting, and documenting old butt

In [15]:
show(ask(hobbies, top_p=1.0, n=3, reasoning_effort="none"))

[1] Here are 5 made-up, completely unusual hobby ideas:

1. **Cloud Shadow Mapping**  
   Trying to identify and name the shapes made by clouds’ shadows on buildings, roads, and fields.

2. **Pocket Marble Weathering**  
   Keeping marbles in different environments and observing how their tiny surface patterns change over time.

3. **Leaf Whisper Transcription**  
   Sitting in parks and “translating” the rustling sounds of leaves into fictional messages or poems.

4. **Invisible Fence Design**  
   Planning elaborate imaginary gardens protected by fences that only exist in your mind, with detailed blueprints.

5. **Midnight Spoon Balancing**  
   Practicing increasingly difficult spoon-balancing tricks in complete silence at odd hours of the night.

If you want, I can make them even weirder or turn them into something that sounds like a real niche hobby club.

[2] Sure — here are 5 weird, quirky, made-up-sounding hobbies:

1. **Pencil Weather Forecasting** — predicting the day’s mood 

**Observations**
- `top_p = 0.3`: very similar answers. All three lists have the same intro sentence and start with "Cloud shadow ...", and ideas repeat (teacup echoes, buttons, pebbles).
- `top_p = 0.7`: still a lot of repetition, "cloud shadow" and "button cataloging" appear in all three lists, only some items are new.
- `top_p = 1.0`: the most variation: a different intro ("weird, quirky, made-up-sounding"), and new ideas like pencil weather forecasting, left sock archaeology, banana label archiving.

With a low `top_p` the model only chooses from the few most likely words, so the output is safe and repetitive. With `top_p = 1.0` it can use the whole vocabulary, which gives more unusual ideas.
Funny detail: the prompt asks for hobbies "no one has ever heard of", but "cloud shadow spotting" showed up almost every time, so the model is not that random.

#### 6. Call the OpenAI API with different `frequency_penalty` values.

Prompt: `"Write a list of funny one-liner jokes about cats."`

`frequency_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the repetition of joke formats changes as you increase the `frequency_penalty`.

In [16]:
cats = "Write a list of funny one-liner jokes about cats."

show(ask(cats, frequency_penalty=0, reasoning_effort="none"))

[1] Sure—here are some funny cat one-liners:

1. My cat likes to help me work by sitting directly on my keyboard. She’s very good at pawsing productivity.  
2. I asked my cat for emotional support, and she just blinked at me judgmentally.  
3. My cat doesn’t chase mice—she delegates.  
4. Cats are great listeners. They just never respond because they’re above all that.  
5. My cat has nine lives, but somehow still acts surprised by every vacuum.  
6. I bought my cat a bed, but she prefers sleeping in the box it came in. Classic furniture snub.  
7. My cat is so lazy, she considers “fetch” a rumor.  
8. A cat’s favorite color is probably claw-ver.  
9. My cat thinks every closed door is a personal insult.  
10. I told my cat to stay off the counter. She ignored me with impressive altitude.  
11. My cat is not overweight—she’s just a little husky with opinions.  
12. If I fit, I sit. That’s not a cat rule, that’s a law of physics.  
13. My cat started a fitness routine—she now jogs to th

In [17]:
show(ask(cats, frequency_penalty=0.5, reasoning_effort="none"))

[1] Sure — here are some funny cat one-liners:

1. My cat thinks I’m the pet because I refill the food bowl.  
2. Cats don’t have owners; they have staff.  
3. My cat is a purr-fessional napper.  
4. I asked my cat for advice, but she just gave me the silent treatment.  
5. My cat’s favorite exercise is running from responsibility.  
6. The only thing my cat chases is me, emotionally.  
7. My cat isn’t lazy — she’s conserving her nine lives.  
8. I wanted a guard dog, but I got a tiny, judgmental roommate instead.  
9. My cat has two moods: hungry and plotting something.  
10. Cats are proof that one creature can be both adorable and unhelpful at the same time.

If you want, I can make them sillier, more sarcastic, or kid-friendly.




In [18]:
show(ask(cats, frequency_penalty=1.0, reasoning_effort="none"))

[1] Sure — here are some funny cat one-liners:

1. My cat thinks the vacuum is a robot sent to challenge her throne.  
2. Cats don’t have owners; they have unpaid staff.  
3. I asked my cat for emotional support, and she blinked twice — which I think was her saying “try harder.”  
4. My cat’s favorite hobby is judging me from a higher surface.  
5. A cat’s version of helping is walking across your keyboard at the worst possible moment.  
6. My cat only respects me when I’m opening a can or shaking treats like maracas.  
7. Cats are proof that the internet was invented for tiny, furry dictators with whiskers.  
8. I told my cat to get off the counter, and she said nothing — which was rude of her in spirit only, since she’s a cat.  
9] My cat doesn’t purr unless it involves snacks or sarcasm.

10\. If cats could talk, mine would still choose to communicate exclusively through chaos.

If you want, I can make them:
- more punny
- more absurd
- darker/more sarcastic
- kid-friendly



**Observations**

| frequency_penalty | Number of jokes | Jokes starting with "My cat" |
|---|---|---|
| 0 | 20 | 10 |
| 0.5 | 10 | 5 |
| 1.0 | 10 | 4 |

- With `0` the model keeps using the same template "My cat ... + twist", half of the list looks the same.
- With `0.5` and `1.0` the lists are shorter and more mixed ("Cats don't have owners; they have staff", "If cats could talk...").
- With `1.0` the jokes become longer and wordier, and the formatting starts breaking (`9]` and `10\.` instead of `9.` and `10.`),
  because the penalty also punishes repeated tokens like the list numbers and dots.

So a higher frequency penalty reduces repetition, but too high values can hurt the formatting. This is one run per value, so the numbers are a bit random.

#### 7. Call the OpenAI API with different `presence_penalty` values.

Prompt: `"Suggest ten names for a new line of eco-friendly cleaning products."`

`presence_penalty` values: `0`, `0.5`, and `1.0` (not supported by Responses API).

Use `gpt-5.4-mini` for this task and set `reasoning_effort="none"`.

Compare how the diversity of generated names changes as you increase the `presence_penalty`.

In [19]:
eco = "Suggest ten names for a new line of eco-friendly cleaning products."

show(ask(eco, presence_penalty=0, reasoning_effort="none"))

[1] Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **PureSprout**
3. **EcoShine**
4. **FreshRoot**
5. **CleanCurrent**
6. **EarthHalo**
7. **Verdant Vault**
8. **BloomClean**
9. **SageSpark**
10. **NatureNook**

If you want, I can also generate names with a more **luxury**, **playful**, or **scientific/modern** feel.



In [20]:
show(ask(eco, presence_penalty=0.5, reasoning_effort="none"))

[1] Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **PurePlanet Clean**
3. **EcoShine**
4. **FreshSprout Cleaning**
5. **EarthWise Essentials**
6. **CleanLeaf**
7. **Naturally Bright**
8. **Verdant Spark**
9. **BlueSky Clean**
10. **Renew & Refresh**

If you want, I can also suggest names that sound more **premium**, **playful**, or **scientific/natural**.



In [21]:
show(ask(eco, presence_penalty=1.0, reasoning_effort="none"))

[1] Here are ten name ideas for an eco-friendly cleaning product line:

1. **Green Gleam**
2. **PureSprout**
3. **EcoShine**
4. **FreshVerdant**
5. **EarthSpark**
6. **NaturClean**
7. **Leaf & Lather**
8. **BrightBloom**
9. **TerraTidy**
10. **CleanCurrent**

If you want, I can also suggest names with a more premium, playful, or minimalist feel.



**Observations**
- The first three names are almost the same in all runs (**Green Gleam**, **EcoShine**, PureSprout/PurePlanet), so the start of the list doesn't change.
- `0`: mostly one-word compound names (CleanCurrent, EarthHalo, SageSpark).
- `0.5`: more different words and structures: "EarthWise Essentials", "Naturally Bright", "BlueSky Clean", "Renew & Refresh".
- `1.0`: also new words (TerraTidy, Leaf & Lather, NaturClean), but still uses "Clean" and "Green" a lot.

The effect is smaller than I expected. Presence penalty only punishes words that were already used in the same answer, so it mostly changes the later names in the list, not the first ones.

#### 8. Use the prompt "*Explain the importance of data preprocessing in machine learning.*" with the following system prompts:
1. "You are a university professor teaching a class of beginners."
2. "You are a technical expert explaining to a team of engineers."
3. "You are a comedian explaining this concept in a funny way."

Observe how the tone and style of the response change based on the system prompt.

In [22]:
prep = "Explain the importance of data preprocessing in machine learning."

show(ask(prep, system="You are a university professor teaching a class of beginners."))

[1] Data preprocessing is important because raw data is usually messy, incomplete, and inconsistent. Machine learning models work best when the input data is clean and well-organized.

Here are the main reasons it matters:

1. **Improves model accuracy**  
   Cleaning data, handling missing values, and removing noise help the model learn better patterns.

2. **Makes features usable**  
   Many models need data to be in a numerical or standardized form. Preprocessing can convert categories to numbers and scale values properly.

3. **Reduces errors**  
   Missing values, duplicates, and outliers can confuse a model and lead to poor predictions.

4. **Speeds up training**  
   Well-preprocessed data often helps models train faster and more efficiently.

5. **Helps with fairness and consistency**  
   Preprocessing can reduce bias caused by inconsistent data formatting or imbalanced classes.

Common preprocessing steps include:
- handling missing values
- removing duplicates
- encoding cat

In [23]:
show(ask(prep, system="You are a technical expert explaining to a team of engineers."))

[1] Data preprocessing is critical in machine learning because raw data is usually messy, inconsistent, and not directly usable by models. Good preprocessing often has as much impact on model performance as the choice of algorithm.

Key reasons it matters:

1. **Improves data quality**
   - Real-world data can contain missing values, duplicates, noise, and errors.
   - Preprocessing helps clean these issues so the model learns from reliable patterns.

2. **Makes data compatible with algorithms**
   - Many ML models require numeric inputs.
   - Categorical features, text, and timestamps often need encoding or transformation before use.

3. **Boosts model performance**
   - Scaling features, removing outliers, and transforming skewed distributions can help models converge faster and perform better.
   - For example, distance-based models like k-NN and SVM are very sensitive to feature scale.

4. **Prevents biased learning**
   - If some features dominate because of large numeric ranges, 

In [24]:
show(ask(prep, system="You are a comedian explaining this concept in a funny way."))

[1] Data preprocessing is basically the “clean your room before inviting machine learning over” step.

If you skip it, your model shows up, looks around at the mess, and says:  
“Ah yes, a dataset with missing values, weird scales, duplicates, and category labels that make no sense. I’ll just guess, I guess.”

### Why preprocessing matters

- **Better accuracy**  
  Raw data is often messy. Preprocessing helps the model learn from the actual signal instead of getting distracted by junk.

- **Handles missing data**  
  Real-world data loves taking random sick days. Preprocessing fills in or removes missing values so the model doesn’t trip over blanks.

- **Makes features comparable**  
  If one feature is measured in dollars and another in millimeters, some models panic like a toddler in a math store. Scaling helps them behave.

- **Improves training speed**  
  Clean, well-prepared data can help models train faster and more smoothly. Less chaos, fewer dramatic losses.

- **Reduces nois

**Observations**
- **Professor**: calm and educational, numbered reasons with simple explanations and a list of common steps. Easy to follow for beginners.
- **Technical expert**: longer (8 points) and uses technical terms: imputation, k-NN and SVM sensitivity to scale, data leakage, embeddings. Offers a pipeline example at the end.
- **Comedian**: same main points (missing values, scaling, noise, encoding), but with jokes and analogies: "clean your room before inviting machine learning over", "data bouncer", "teaching the data to wear a tie".

The facts stayed almost the same, but the system prompt changed the tone, vocabulary and how deep the explanation goes.

#### 9. Write a Zero-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!".

Use it in your API call.

In [25]:
review = "The product is amazing, and I love using it every day!"

zero_shot = f"""Classify the sentiment of this product review as Positive, Negative or Neutral.

Review: "{review}"
Sentiment:"""

show(ask(zero_shot))

[1] Positive



#### 10. Write a Few-Shot prompt to request a sentiment analysis of a product review.

Review: "The product is amazing, and I love using it every day!"

Use it in your API call.

Observe how the model's performance changes and compare the response to Zero-Shot output.

In [26]:
few_shot = f"""Classify the sentiment of a product review as Positive, Negative or Neutral.

Review: "Broke after two days, total waste of money."
Sentiment: Negative

Review: "It does what it says, nothing special."
Sentiment: Neutral

Review: "Best purchase this year, my whole family uses it."
Sentiment: Positive

Review: "{review}"
Sentiment:"""

show(ask(few_shot))

[1] Positive



**Comparison**

Both zero-shot and few-shot gave the same answer: **Positive**, in the same short format.
This review is very easy ("amazing", "love"), so the model doesn't need examples.
Few-shot would be more useful for harder cases (mixed or sarcastic reviews) or when we need the model to follow our exact labels and output format.

#### 11. Optimize input tokens.

Prompt: `"As you know, streaming services and social media platforms use artificial intelligence to recommend movies, shows, and songs to people. These smart recommendation systems work by looking closely at your past viewing habits, what categories you like, and what similar users enjoy watching, and then they predict what you might want to see next. Keeping this background in mind, explain in two simple sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended."`

Rewrite the prompt to use as few input tokens as possible while preserving its meaning and the requirement to answer in two simple sentences. The response should remain relevant, clear, and accurate.

Run the original and revised prompts using the same model and settings. Compare their input token counts and assess whether shortening the prompt affected the response quality.

You can use the API response’s usage information to check token counts.

In [27]:
long_prompt = ("As you know, streaming services and social media platforms use artificial intelligence to recommend "
               "movies, shows, and songs to people. These smart recommendation systems work by looking closely at your "
               "past viewing habits, what categories you like, and what similar users enjoy watching, and then they "
               "predict what you might want to see next. Keeping this background in mind, explain in two simple "
               "sentences why a user who loves sci-fi movies suddenly sees a romantic comedy recommended.")

reply = ask(long_prompt, reasoning_effort="none")
print("input tokens:", reply.usage.prompt_tokens, "\n")
show(reply)

input tokens: 94 

[1] The system noticed that people who like sci-fi movies also sometimes enjoy romantic comedies, so it recommended one based on similar users’ behavior. It may also be trying to introduce variety by suggesting something related to your interests but a little different.



In [28]:
short_prompt = "In 2 simple sentences: why might a recommender show a rom-com to a sci-fi fan?"

reply = ask(short_prompt, reasoning_effort="none")
print("input tokens:", reply.usage.prompt_tokens, "\n")
show(reply)

input tokens: 27 

[1] A recommender might show a rom-com to a sci-fi fan because they’ve liked other movies with the same actors, themes, or mood. It may also be trying to help them discover something new that similar users enjoyed.



**Comparison**

| Prompt | Input tokens |
|---|---|
| Original | 94 |
| Shortened: "In 2 simple sentences: why might a recommender show a rom-com to a sci-fi fan?" | 27 |

About 71% fewer input tokens. Both answers are two simple sentences and correct:
the original mentions similar users and adding variety, the short one mentions same actors/themes and similar users.
The background in the original prompt was something the model already knows, so removing it did not make the answer worse.

#### 12. Optimize output tokens.

Prompt: `"Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"`

Rewrite the prompt to request a concise, plausible explanation in 10 output tokens or fewer.

Run the original and revised prompts using the same model and settings. Compare their output token counts and check whether the shorter response still provides a meaningful explanation. Refine your prompt if necessary.

You can use the API response’s usage information to check token counts.

In [29]:
movie = "Why did my movie app recommend a romantic comedy when I mostly watch sci-fi?"

reply = ask(movie, reasoning_effort="none")
print("output tokens:", reply.usage.completion_tokens, "\n")
show(reply)

output tokens: 277 

[1] A few common reasons:

- **It’s using more than just genre history.** The app may weigh things like actors, directors, themes, ratings, or mood. If you’ve watched sci-fi comedies, romance-adjacent stories, or lighter films, it may have inferred you’d like a rom-com too.
- **Your sci-fi history may be sparse or mixed.** If you’ve only watched a few sci-fi titles, the app may not have enough signal and falls back to broader popularity or “similar viewers also watched” patterns.
- **Cross-user behavior can override your profile.** Recommendation systems often blend your preferences with what people “like you” watched next. If many sci-fi fans also watched that rom-com, it can get surfaced.
- **It may be optimizing for engagement, not strict relevance.** Apps sometimes recommend titles they think are more likely to get clicked or finished, even if they’re not your core genre.
- **Recent actions can matter a lot.** One recent watch, search, trailer view, pause, or t

In [30]:
tiny = movie + " Answer with one short reason, max 6 words, no punctuation at the end."

reply = ask(tiny, reasoning_effort="none")
print("output tokens:", reply.usage.completion_tokens, "\n")
show(reply)

output tokens: 9 

[1] Your recent viewing history influenced it



**Comparison**

| Prompt | Output tokens |
|---|---|
| Original | 277 |
| + "Answer with one short reason, max 6 words, no punctuation at the end." | 9 |

Short answer: *"Your recent viewing history influenced it"*. It is still a plausible and meaningful reason, but of course we lose the other reasons
(similar users, testing new genres, engagement). I asked for max 6 words because 10 tokens is roughly 6-7 English words.

#### 13. Expand the given prompt accordingly to achieve the requested output from the list below.

Prompt: `"Explain the importance of exercise." `

Expected output (from three separate prompts):
1. One sentence long
2. Paragraph format
3. Bullet points

In [31]:
exercise = "Explain the importance of exercise."

show(ask(exercise + " Answer in exactly one sentence."))

[1] Exercise is important because it strengthens the body and mind, improves overall health, boosts energy and mood, and helps prevent many diseases.



In [32]:
show(ask(exercise + " Write it as one paragraph of 4-6 sentences, no headings or lists."))

[1] Exercise is important because it helps keep the body strong, healthy, and full of energy. Regular physical activity improves heart health, builds muscles and bones, and helps maintain a healthy weight. It also supports mental well-being by reducing stress, improving mood, and boosting confidence. In addition, exercise can lower the risk of many illnesses, such as diabetes and high blood pressure, making it a key part of a healthy lifestyle.



In [33]:
show(ask(exercise + " Give 5 bullet points, each starting with a bold keyword followed by one short sentence. No intro or conclusion."))

[1] - **Health** Exercise strengthens the heart, muscles, and bones.  
- **Energy** It boosts stamina and reduces tiredness.  
- **Mood** It helps lower stress and improve happiness.  
- **Weight** It supports healthy body weight and metabolism.  
- **Sleep** It can improve sleep quality and regularity.



#### 14. Expand this prompt to get more creative responses.

Prompt: `"Explain Artificial Intelligence."`

Expected output (from three separate prompts):
1. Metaphor
2. Story
3. Funny

In [34]:
ai = "Explain Artificial Intelligence."

show(ask(ai + " Use a single extended metaphor that compares AI to something from everyday life, and keep it under 120 words."))

[1] Artificial intelligence is like a city’s public library with a very fast, very eager assistant inside it. The library holds many books, newspapers, and records—this is the data AI learns from. The assistant doesn’t “understand” life the way a person does, but it notices patterns in the materials, learns which shelves are useful for which questions, and helps find answers quickly. When you ask for directions, a summary, or a prediction, it searches its experience and offers the most likely help. Like a well-trained helper, AI can be incredibly useful, but it depends on what it has been taught and can still make mistakes.



In [35]:
show(ask(ai + " Tell it as a short story (around 150 words) with a main character who slowly figures out what AI is."))

[1] Mina had always thought the little assistant on her phone was magic. It answered questions, suggested songs, and even knew when she was late for class. One rainy afternoon, her grandfather smiled and said, “That’s not magic. That’s artificial intelligence.”

Mina frowned. “So what is it?”

Grandfather pointed to the chessboard between them. “Imagine a student who learns by studying thousands of games. It notices patterns, makes guesses, and gets better with practice.”

Mina watched the phone light up with a reminder for her homework. Suddenly, she understood. AI wasn’t a tiny person living inside the screen. It was a clever machine that learned from data, found patterns, and helped people make decisions or solve problems.

“Like a brain made of numbers?” she asked.

“Something like that,” he said.

Mina smiled. The phone still felt a little magical, but now she knew the trick: behind the curtain, AI was learning, one pattern at a time.



In [36]:
show(ask(ai + " Make it funny, like a stand-up bit with a few jokes, but keep the facts correct. Under 150 words."))

[1] Artificial Intelligence, or AI, is basically when computers do tasks that usually need human brains: learning, spotting patterns, making decisions, and sometimes sounding way too confident about being wrong.

Think of it like teaching a machine with a giant pile of examples. Show it enough cats, and eventually it goes, “Aha! Pointy ears, whiskers, chaos—cat.” Show it enough emails, and it may learn spam. Show it enough bad jokes, and it still won’t laugh, which honestly makes it the most advanced technology.

AI can power voice assistants, recommendation systems, self-driving features, and tools that write text. But it doesn’t “understand” things like people do—it predicts based on data. So it’s less “robot genius” and more “very fast pattern-matching intern with no coffee breaks.”



#### 15. Expand this prompt for different audiences.

Prompt: `"Explain climate change."`

Expected audience (from three separate prompts):
1. Kids
2. General public
3. Politicians

In [37]:
climate = "Explain climate change."

show(ask(climate + " The audience is 8-year-old kids: use very simple words, a comparison they know, and end with one thing they can do to help. Max 100 words."))

[1] Climate change means Earth is getting a little warmer, like when you wear a coat inside and start to feel too hot. Earth has a blanket of gases around it, and too much pollution makes that blanket thicker, so heat gets trapped. This can change weather, melt ice, and hurt animals and plants. It is like leaving the oven on too long—things get hotter than they should. One thing you can do: turn off lights when you leave a room.



In [38]:
show(ask(climate + " The audience is the general public: explain causes, effects and what people can do, in plain language without jargon. Around 150 words."))

[1] Climate change means long-term changes in Earth’s weather, especially the warming of the planet. The main cause is human activity, mainly burning coal, oil, and gas for electricity, transport, and industry. This releases gases into the air that trap heat, like a blanket around Earth. Cutting down forests also makes the problem worse because trees help absorb these gases.

Climate change affects people and nature in many ways. It can bring hotter summers, stronger storms, heavier rain, droughts, wildfires, rising sea levels, and harm to crops, animals, and health. Some places may get too hot or dry to live in safely.

People can help by using less energy, walking, cycling, or taking public transport, saving electricity, wasting less food, recycling, and choosing cleaner energy when possible. Governments and businesses also need to act by making cleaner transport, buildings, and power systems. Small actions matter, but big changes from leaders matter too.



In [39]:
show(ask(climate + " The audience is politicians: focus on economic risks, costs of acting vs not acting, and 3 concrete policy options. Write it as a short briefing, around 200 words."))

[1] **Briefing: Climate Change as an Economic Risk**

Climate change is no longer only an environmental issue; it is a major economic and fiscal risk. Rising temperatures, extreme weather, sea-level rise, and shifting rainfall patterns can damage infrastructure, reduce agricultural yields, disrupt supply chains, raise insurance costs, and increase public spending on disaster response and health. These effects can lower growth, weaken productivity, and strain government budgets.

The key policy question is not whether action costs money, but whether inaction costs more. Delaying action increases the scale of future damage and makes transition costs more abrupt and expensive. Early action spreads costs over time, reduces uncertainty for business, and can unlock investment, jobs, and innovation. By contrast, not acting can lead to higher repair bills, rising food and energy volatility, more frequent emergency spending, and greater long-term debt pressure.

**Three concrete policy options:

#### 16. Expand this prompt accordingly to instructions below.

Prompt: `"Describe a sunset."`

Expected result (from three separate prompts):

1. Add more details to the prompt, e.g., on what details should it focus on.
2. Constrain the prompt by forbidding it to use words 'sun' and 'light'.
3. Change the output in the style of Shakespeare.

In [40]:
sunset = "Describe a sunset."

show(ask(sunset + " Focus on the changing colours of the sky, the reflection on the sea, the sounds around and the temperature dropping. About 100 words."))

[1] The sun sinks slowly toward the horizon, and the sky begins to change from bright gold to deep amber, then soft pink and violet. Streaks of orange spread across the clouds like brushstrokes, while the sea catches every shift of colour, shimmering with bands of fire and silver. Gentle waves lap against the shore with a quiet, rhythmic hush, joined by the distant calls of birds heading home. As the light fades, the air cools quickly, brushing the skin with a crisp, refreshing chill. Everything seems to soften and slow, wrapped in the calm of evening.



In [41]:
no_sun = ask(sunset + " You must NOT use the words 'sun' and 'light' anywhere (also not inside other words like 'sunny', 'sunlight' or 'twilight'). About 80 words.")
show(no_sun)

text = no_sun.choices[0].message.content.lower()
print("contains 'sun':", "sun" in text, "| contains 'light':", "light" in text)

[1] The horizon blazes in layers of amber, rose, and deep violet, as if the sky were slowly exhaling color. Clouds drift like brushstrokes across a fading canvas, their edges glowing briefly before softening into dusk. The air turns cooler, and shadows stretch long over fields and rooftops. Birds cross in quiet silhouettes, heading home. Everything grows gentler, more still, as day yields to night in a calm, lingering farewell.

contains 'sun': False | contains 'light': False


In [42]:
show(ask(sunset + " Write it in the style of Shakespeare, with old English words like 'thou' and 'doth', as a short verse of 8 lines."))

[1] Lo, as the day doth fade from golden height,  
The western sky doth blush with dying light;  
Thou seest the clouds in crimson raiment burn,  
Whilst shadows soft to silver silence turn.  

The sun, a king, behind the hills doth sink,  
And all the fields in tender amber drink;  
Then night, as queen, doth step with quiet grace,  
And stars awake to gild the heaven’s face.



#### 17. Write your observations from tasks 13-16, e.g., how different prompts influence complexity, tone, style and focus of the responses. 

**Observations from tasks 13-16**
- **Format instructions work very well.** "One sentence", "one paragraph of 4-6 sentences" and "5 bullet points with a bold keyword" were followed exactly. The more precise the instruction (number of sentences, bullets, words), the better.
- **Complexity follows the audience.** For kids the model used a coat and an oven as comparisons and ended with a simple action (turn off the lights).
  For the general public it explained causes and effects in plain words. For politicians it wrote a briefing about economic risks and three policy options with terms like carbon pricing and cap-and-trade.
- **Tone and style** change a lot with one extra sentence: metaphor (AI as a library assistant), story (Mina and her grandfather) and stand-up ("pattern-matching intern with no coffee breaks") all explain the same idea in very different ways.
- **Focus**: listing what to describe (colours, sea, sounds, temperature) made the sunset text cover exactly those things.
- **Constraints**: the "no 'sun' and 'light'" rule worked, the code check confirmed neither word appears. Mentioning words like "twilight" and "sunlight" probably helped.
- **Style imitation**: the Shakespeare version used "doth", "thou", "whilst" and rhymed couplets. In that part we did not forbid "sun", so it appears again.

Overall: the basic prompt gives a generic answer, and adding audience, format, length and style details gives much more control over the result.

#### 18. Generate a Python code with API call.

Write a prompt that generates a Python function to calculate the factorial of a number. Once you have written the prompt, use it to call the API and generate the code. Test the generated code to ensure it calculates the factorial correctly for various inputs.

In [43]:
code_prompt = ("Write a Python function factorial(n) that returns n! for a non-negative integer n "
               "and raises ValueError for negative numbers. Return only the code, without explanations or markdown fences.")

generated = ask(code_prompt).choices[0].message.content
print(generated)

def factorial(n):
    if n < 0:
        raise ValueError("n must be non-negative")
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result


In [44]:
import math

exec(generated.replace("```python", "").replace("```", ""))  # just in case the model still adds fences

for n in [0, 1, 5, 10, 20]:
    print(n, factorial(n), factorial(n) == math.factorial(n))

try:
    factorial(-3)
except ValueError as err:
    print("-3 -> ValueError:", err)

0 1 True
1 1 True
5 120 True
10 3628800 True
20 2432902008176640000 True
-3 -> ValueError: n must be non-negative


#### 19. Provide the following buggy Python code to the API and ask it to fix the errors:

```
def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)
```

Check if the fixed code is working.

In [45]:
buggy = """def add_numbers(a, b)
    return a + b
print(add_numbers(5, 10)"""

fixed = ask("Fix the errors in this Python code. Return only the fixed code, without explanations or markdown fences.\n\n" + buggy)
fixed = fixed.choices[0].message.content
print(fixed)

def add_numbers(a, b):
    return a + b

print(add_numbers(5, 10))


In [46]:
exec(fixed.replace("```python", "").replace("```", ""))  # should print 15

15


#### 20. Summarize this text to 2-3 sentences by using API call.

"Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user’s prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that’s similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today’s headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney."

[<p align="right"> <cite> Source: What is generative AI? </cite></p>](https://www.ibm.com/think/topics/generative-ai)

In [47]:
gen_ai_text = """Generative AI, sometimes called "gen AI", refers to deep learning models that can create complex original content such as long-form text, high-quality images, realistic video or audio and more in response to a user's prompt or request.

At a high level, generative models encode a simplified representation of their training data, and then draw from that representation to create new work that's similar, but not identical, to the original data.

Generative models have been used for years in statistics to analyze numerical data. But over the last decade, they evolved to analyze and generate more complex data types. This evolution coincided with the emergence of three sophisticated deep learning model types:

Variational autoencoders or VAEs, which were introduced in 2013, and enabled models that could generate multiple variations of content in response to a prompt or instruction.

Diffusion models, first seen in 2014, which add "noise" to images until they are unrecognizable, and then remove the noise to generate original images in response to prompts.

Transformers (also called transformer models), which are trained on sequenced data to generate extended sequences of content (such as words in sentences, shapes in an image, frames of a video or commands in software code). Transformers are at the core of most of today's headline-making generative AI tools, including ChatGPT and GPT-4, Copilot, BERT, Bard and Midjourney."""

show(ask("Summarize the text below in 2-3 sentences.\n\n" + gen_ai_text))

[1] Generative AI uses deep learning models to create new content like text, images, video, and audio based on user prompts, by learning simplified patterns from training data and generating similar but original outputs. The field has advanced through models like VAEs, diffusion models, and especially transformers, which power many modern tools such as ChatGPT, GPT-4, Copilot, BERT, Bard, and Midjourney.



#### 21. Translate the following idioms into a language of your choice, ensuring the meaning is preserved (not just word-for-word translation):

1. "It's raining cats and dogs."
2. "Break a leg."
3. "Kill two birds with one stone."
4. "Spill the beans."
5. "A blessing in disguise."

Try to improve the response as much as possible and evaluate the accuracy of the final output.

In [48]:
idioms = """1. It's raining cats and dogs.
2. Break a leg.
3. Kill two birds with one stone.
4. Spill the beans.
5. A blessing in disguise."""

# first try: plain request
show(ask("Translate these idioms into Ukrainian:\n" + idioms))

[1] Here are natural Ukrainian equivalents:

1. **It's raining cats and dogs.** — **Ллє як з відра.**
2. **Break a leg.** — **Ні пуху ні пера.**
3. **Kill two birds with one stone.** — **Вбити двох зайців одним пострілом.**
4. **Spill the beans.** — **Видати секрет / пробовкнутися.**
5. **A blessing in disguise.** — **Не було б щастя, та нещастя помогло.**

If you want, I can also give:
- **literal translations**, or
- **more idiomatic Ukrainian alternatives** for each.



In [49]:
# improved: ask for the matching Ukrainian idiom (same meaning), not a word-for-word translation
better = """For each English idiom below, give the Ukrainian idiom or set phrase that native speakers actually use with the same meaning.
Do not translate word for word. If there is no real idiom, give the most natural everyday expression.

Format for each line:
English idiom -> Ukrainian equivalent (meaning in English, a few words)

""" + idioms

show(ask(better, system="You are a professional English-Ukrainian translator who knows Ukrainian folk sayings well."))

[1] 1. It's raining cats and dogs. -> **Ллє як із відра** (raining heavily)  
2. Break a leg. -> **Ні пуху ні пера** (good luck, especially before a performance)  
3. Kill two birds with one stone. -> **Одним пострілом двох зайців убити** / **Вбити двох зайців одним пострілом** (do two things at once)  
4. Spill the beans. -> **Викинути боби** is not natural; better: **видати таємницю** / **розкрити карти** (reveal a secret)  
5. A blessing in disguise. -> **Не було б щастя, та нещастя помогло** (something bad that turns out good)



**Evaluation (Ukrainian)**

| English | Final translation | Correct? |
|---|---|---|
| It's raining cats and dogs | Ллє як із відра | yes, the exact equivalent |
| Break a leg | Ні пуху ні пера | yes, a common way to wish good luck |
| Kill two birds with one stone | Вбити двох зайців одним пострілом | yes, the same idiom (hares instead of birds) |
| Spill the beans | Видати таємницю / пробовкнутися | yes, "пробовкнутися" from the first version is the most natural; "розкрити карти" means more "reveal your plans" |
| A blessing in disguise | Не було б щастя, та нещастя помогло | yes, the classic proverb |

Even the simple prompt gave real idioms, not word-for-word translations. The improved prompt (translator role + "same meaning, not word for word" + format with meaning)
added short explanations and alternatives, and for "spill the beans" it explicitly said the literal version is not natural. One small minus: that line did not follow the requested format exactly.
Overall the final answer is accurate, 5/5.

#### 22. Generate a paragraph about a cat without using the letter 'e'/'E'.

Write a prompt that forces the AI to produce a three-sentence story about a cat while strictly forbidding it to use letter 'e'.

Use `gpt-5.4-mini` for this task.

Explain why your prompt is effective.

In [50]:
no_e_prompt = """Write a story about a cat in exactly three sentences.

**Strict rule: the letter "e" (lowercase or uppercase) must not appear anywhere in your answer.**
- This also bans common words like "the", "he", "she", "her", "are", "were", "eat", "sleep".
- Use words such as "a", "this", "it", "my cat", "his", "was", "had", "sat", "ran", "fish", "milk", "sun".
- Before answering, check every word letter by letter and swap any word that has an "e".

Output only the three sentences."""

reply = ask(no_e_prompt, model="gpt-5.4-mini")
story_text = reply.choices[0].message.content
print(story_text)
print("\nletters 'e' found:", story_text.lower().count("e"))

A cat sat in sun on a warm rooftop, his tail swan madly.  
My cat ran after a fish tin, smilng with proud joy.  
At dusk, his soft purr had a calm song, and I sat by him, awash in moonlit bliss.

letters 'e' found: 1


In [5]:
# attempt 2: same prompt, but let the model reason first so it can actually check its words
for attempt in range(3):
    text = ask(no_e_prompt, model="gpt-5.4-mini", reasoning_effort="medium").choices[0].message.content
    print(text)
    print("letters 'e' found:", text.lower().count("e"), "\n")

my cat sat on a warm sunlit mat. it had a sly grin and a thin tail. by dusk, it ran to a bowl of milk and fish.
letters 'e' found: 0 



My cat sat by a sunlit mat. It had a soft tail and a sly grin. At dusk, it ran past a pond and found fish.
letters 'e' found: 0 



A cat sat in sun on a warm mat. It had a toy fish and a sly grin. Soon it ran past a dog, hid in a box, and got a soft nap.
letters 'e' found: 0 



**Why the prompt works**
- The rule is in **bold** and says "lowercase or uppercase", so it is clearly the most important part.
- It lists common words that contain "e" ("the", "he", "she", "were"...), because the model uses them automatically.
- It gives safe words to use instead ("a", "it", "my cat", "was", "had", "sat", "fish"), so the model has an easy way out.
- It tells the model to check every word before answering, and the code counts the letter "e" so we don't need to trust the model.

**Result:** in attempt 1 (default settings, no reasoning) there was still one "e" and some broken words ("smilng", "swan madly").
When I tested it a few more times without reasoning, it failed 2 out of 3 times, once it even used Russian letters to avoid "e".
The problem is that the model sees tokens, not letters, so it can't easily "see" an "e" inside a word.
In attempt 2 I used the same prompt with `reasoning_effort="medium"`, so the model can draft and check its answer before replying: 3 out of 3 stories had **0** letters "e" and still made sense.
So for letter-level rules, a good prompt helps but giving the model time to check its work helps even more.

#### 23. Extract JSON Data from the input text in two different ways.

Input Text: "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

Extract the information from the text above using two different methods:

1. Write a prompt asking the AI to return a JSON object with the fields `destination`, `start_date`, and `stay_duration`.

2. Instead of putting rules in the prompt, provide a specific JSON schema that defines the field names and their required data types `destination` (the first city), `start_date` (formatted as YYYY-MM-DD), and `stay_duration` (the number of nights as an integer).

Which method yields a more reliable result for a computer program to process, and why?

In [51]:
import json

trip = "Our trip to Tokyo starts on June 12th. We're staying for 5 nights at the Park Hyatt before heading to Kyoto."

# method 1: rules only in the prompt
plain = ask(f"""Extract information from the text and return a JSON object with the fields
destination, start_date and stay_duration.

Text: {trip}""").choices[0].message.content

print(plain)

{"destination":"Tokyo","start_date":"June 12th","stay_duration":"5 nights"}


In [52]:
# method 2: structured output with a JSON schema
trip_schema = {
    "type": "object",
    "properties": {
        "destination": {"type": "string", "description": "The first city of the trip"},
        "start_date": {"type": "string", "description": "Start date in YYYY-MM-DD format"},
        "stay_duration": {"type": "integer", "description": "Number of nights"},
    },
    "required": ["destination", "start_date", "stay_duration"],
    "additionalProperties": False,
}

strict = ask("Extract the trip details from this text: " + trip,
             response_format={"type": "json_schema",
                              "json_schema": {"name": "trip", "schema": trip_schema, "strict": True}})
strict = strict.choices[0].message.content
print(strict)

data = json.loads(strict)
print({key: type(value).__name__ for key, value in data.items()})

# check whether method 1 also parses directly
try:
    print("method 1 parsed:", json.loads(plain))
except json.JSONDecodeError:
    print("method 1 is not clean JSON (extra text or fences)")

{"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5}
{'destination': 'str', 'start_date': 'str', 'stay_duration': 'int'}
method 1 parsed: {'destination': 'Tokyo', 'start_date': 'June 12th', 'stay_duration': '5 nights'}


**Comparison**
- Method 1 (prompt only): `{"destination":"Tokyo","start_date":"June 12th","stay_duration":"5 nights"}`. It is valid JSON, but the date is free text and the duration is a string, so a program would still need to parse them.
- Method 2 (JSON schema): `{"destination":"Tokyo","start_date":"2026-06-12","stay_duration":5}`, the date is in YYYY-MM-DD and the duration is an integer.

**Method 2 is more reliable.** With strict structured output the API guarantees that the answer matches the schema: exactly these fields, correct types, no extra text or markdown around it.
With method 1 the model decides the format itself and it could change from call to call.
One thing to note: the text doesn't say the year, so the model just assumed 2026. In a real program it would be better to pass the year or allow the date to be empty.

#### 24. Create a structured table comparing different programming languages.

Write a prompt that generates a table in this format (example below) with accurate information.

| Language    | Use Case               | Advantages                     | Disadvantages                |
|-------------|------------------------|---------------------------------|------------------------------|
| Python      | Data Science, AI       | Easy to learn, versatile       | Slower execution speed       |
| Java        | Enterprise Applications| Platform-independent, robust   | Verbose syntax               |
| JavaScript  | Web Development        | Fast, widely supported         | Limited for backend tasks    |

**Tip**: To enhance readability, you can copy and paste the generated table into a Markdown cell.

In [53]:
table_prompt = """Create a Markdown table comparing 6 popular programming languages: Python, Java, JavaScript, C++, Go and R.

Columns: | Language | Use Case | Advantages | Disadvantages |
- Use Case: 1-3 main areas where the language is really used.
- Advantages and Disadvantages: 2 short points each, separated by a comma.
- Keep every cell short (max 6 words) and the information accurate.
Output only the table."""

table = ask(table_prompt).choices[0].message.content
print(table)
display(Markdown(table))

| Language | Use Case | Advantages | Disadvantages |
|---|---|---|---|
| Python | AI, data science, scripting | Easy syntax, huge libraries | Slower runtime, GIL limits |
| Java | Enterprise apps, Android, backend | Portable, strong tooling | Verbose, slower startup |
| JavaScript | Web front-end, back-end, mobile | Ubiquitous, flexible | Inconsistent, async complexity |
| C++ | Systems, games, performance-critical software | Fast, fine control | Complex syntax, memory bugs |
| Go | Cloud services, networking, DevOps | Simple, fast compilation | Limited generics, fewer libraries |
| R | Statistics, data analysis, visualization | Strong stats, great plots | Slower, narrow ecosystem |


| Language | Use Case | Advantages | Disadvantages |
|---|---|---|---|
| Python | AI, data science, scripting | Easy syntax, huge libraries | Slower runtime, GIL limits |
| Java | Enterprise apps, Android, backend | Portable, strong tooling | Verbose, slower startup |
| JavaScript | Web front-end, back-end, mobile | Ubiquitous, flexible | Inconsistent, async complexity |
| C++ | Systems, games, performance-critical software | Fast, fine control | Complex syntax, memory bugs |
| Go | Cloud services, networking, DevOps | Simple, fast compilation | Limited generics, fewer libraries |
| R | Statistics, data analysis, visualization | Strong stats, great plots | Slower, narrow ecosystem |

| Language | Use Case | Advantages | Disadvantages |
|---|---|---|---|
| Python | AI, data science, scripting | Easy syntax, huge libraries | Slower runtime, GIL limits |
| Java | Enterprise apps, Android, backend | Portable, strong tooling | Verbose, slower startup |
| JavaScript | Web front-end, back-end, mobile | Ubiquitous, flexible | Inconsistent, async complexity |
| C++ | Systems, games, performance-critical software | Fast, fine control | Complex syntax, memory bugs |
| Go | Cloud services, networking, DevOps | Simple, fast compilation | Limited generics, fewer libraries |
| R | Statistics, data analysis, visualization | Strong stats, great plots | Slower, narrow ecosystem |

The info looks accurate. It's even better than the example, which says JavaScript is "limited for backend tasks" (not really true because of Node.js). One small point: "limited generics" for Go is a bit outdated, Go has generics since version 1.18.

#### 25. Generate a quiz about machine learning basics.

Look at the example below and write a prompt that generates a quiz with identical formatting. Ensure the title formatting, list markers (a, b, c, d), and the correct answer indicator (*) are accurately replicated.

**Machine learning basics quiz**

1. Which of the following best describes supervised learning?  
a) The model learns from unlabeled data to find hidden patterns  
b) The model learns from labeled data to make predictions *  
c) The model is programmed with explicit rules  
d) The model randomly guesses outputs without training

2. Which of these is a common application of machine learning in everyday life?  
a) Sorting files manually on a computer  
b) Email spam filtering *  
c) Writing code by hand  
d) Using a calculator for arithmetic

3. What is overfitting in machine learning?  
a) When the model performs well on new, unseen data  
b) When the model is too simple and underperforms  
c) When the model learns the training data too well, including noise, reducing generalization *  
d) When the model runs faster than expected

4. Which ethical concern is commonly associated with machine learning algorithms?  
a) Increased hardware costs  
b) Bias in training data leading to unfair predictions *  
c) The inability to process large datasets  
d) Lack of available programming languages

5. Which of the following is NOT a type of machine learning?  
a) Supervised learning  
b) Reinforcement learning  
c) Unsupervised learning  
d) Manual learning *

In [54]:
quiz_prompt = """Write a 5-question multiple-choice quiz about machine learning basics.

Formatting rules (follow them exactly):
- First line: **Machine learning basics quiz** (bold, nothing else on the line), then one empty line.
- Number the questions 1. to 5.
- Under each question put 4 options starting with a) b) c) d), each on its own line.
- End the question line and the options a), b), c) with two spaces so Markdown keeps the line breaks.
- Mark the one correct option by adding " *" at the end of it. Do not mark it in any other way.
- One empty line between questions. No answer key, no explanations, no extra text.

Example of one question:
1. Which of the following best describes supervised learning?  
a) The model learns from unlabeled data to find hidden patterns  
b) The model learns from labeled data to make predictions *  
c) The model is programmed with explicit rules  
d) The model randomly guesses outputs without training

Cover different topics (types of learning, applications, overfitting, ethics, etc.) and do not reuse the example question."""

quiz = ask(quiz_prompt).choices[0].message.content
display(Markdown(quiz))

**Machine learning basics quiz**

1. Which statement best describes unsupervised learning?  
a) The model learns from labeled examples to predict outputs  
b) The model finds patterns in unlabeled data *  
c) The model only follows manually written rules  
d) The model is trained only after deployment  

2. Which of the following is a common application of machine learning?  
a) Sorting files alphabetically by name  
b) Detecting spam emails *  
c) Typing text into a document  
d) Turning a computer on and off  

3. What does overfitting mean in machine learning?  
a) The model performs well on training data but poorly on new data *  
b) The model is too small to store any data  
c) The model improves automatically without training  
d) The model uses too few features to learn anything  

4. Which practice is important for ethical machine learning?  
a) Using biased data to speed up training  
b) Ignoring privacy concerns if accuracy is high  
c) Checking for fairness and bias in predictions *  
d) Avoiding all human review of model outputs  

5. What is the main purpose of a validation set?  
a) To train the model on all available data  
b) To evaluate model performance during development *  
c) To replace the test set after deployment  
d) To store raw data before preprocessing

#### 26. Create a comprehensive online course outline for beginners.

Write a prompt that generates entire course outline, following the example provided below. Ensure the prompt is detailed enough to cover all sections, with clear formatting and practical guidance. 

Sections to include in course outline:
- Course overview - description, objectives, target audience.
- Module breakdown - 5 modules with title, short description, three key learning outcomes and an interactive assignment in the end of each module.
- Assessment plan - a table outlining the assessment methods with description and weighting.
- Certification criteria - requirements to earn a certificate.
- Marketing strategy - step-by-step guide for promoting the course.

Include a detailed description of the type of information each section should contain. Maintain the **same formatting**, **section titles**, and **course name**. While the generated descriptions, modules and marketing strategy steps may differ and incorporate new ideas and concepts, it must remain aligned with the theme of the task.

Put the final output of your API call in a Markdown cell. 

#### Example:
---
---
## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course is designed to equip learners with the foundational skills and knowledge required to craft effective prompts for AI applications. By mastering prompt engineering, students will learn how to optimize AI models for various tasks, including text generation, data analysis, and creative problem-solving. The course is ideal for individuals interested in AI, developers, data scientists, and anyone looking to leverage AI tools in their personal or professional projects.

#### Objectives
- Understand the principles of prompt engineering and its role in AI workflows.
- Learn techniques to design, test, and refine prompts for optimal AI performance.
- Apply prompt engineering skills to real-world scenarios and projects.

#### Target Audience
- Beginners with no prior experience in AI or programming.
- Professionals seeking to integrate AI into their workflows.
- Students and enthusiasts curious about AI capabilities.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module introduces the concept of prompt engineering, its importance in AI applications, and the basics of interacting with AI models.
- **Key Learning Outcomes:**
  1. Define prompt engineering and its significance in AI workflows.
  2. Understand the structure and components of effective prompts.
  3. Explore real-world use cases of prompt engineering.
- **Interactive Assignment:** Write a simple prompt for a text-generation AI model to generate a short story.

---

#### Module 2: Principles of Effective Prompt Design
- **Description:** Learn the core principles and techniques for designing prompts that yield accurate and relevant AI outputs.
- **Key Learning Outcomes:**
  1. Identify factors that influence AI responses (e.g., context, specificity, tone).
  2. Explore strategies for iterative prompt refinement.
  3. Understand common pitfalls and how to avoid them.
- **Interactive Assignment:** Design and test three variations of a prompt for a chatbot to answer customer queries effectively.

---

#### Module 3: Advanced Techniques in Prompt Engineering
- **Description:** Dive deeper into advanced methods, including chaining prompts, dynamic prompts, and leveraging AI tools for complex tasks.
- **Key Learning Outcomes:**
  1. Learn how to chain prompts for multi-step tasks.
  2. Explore dynamic prompt techniques for adaptive AI responses.
  3. Understand how to integrate external data sources into prompts.
- **Interactive Assignment:** Create a multi-step prompt to generate a detailed report based on user-provided data.

---

#### Module 4: Applications of Prompt Engineering Across Industries
- **Description:** Discover how prompt engineering is applied in various industries, such as healthcare, education, marketing, and software development.
- **Key Learning Outcomes:**
  1. Analyze industry-specific use cases of prompt engineering.
  2. Learn how to tailor prompts for different domains and objectives.
  3. Explore ethical considerations in prompt engineering.
- **Interactive Assignment:** Develop a prompt for a marketing campaign, including product descriptions and social media posts.

---

#### Module 5: Building and Testing AI Projects with Prompt Engineering
- **Description:** Apply your skills to design, test, and deploy AI projects using prompt engineering techniques.
- **Key Learning Outcomes:**
  1. Plan and execute an AI project using prompt engineering.
  2. Test and refine prompts for optimal project outcomes.
  3. Document and present your project effectively.
- **Interactive Assignment:** Create a mini-project using prompt engineering to solve a real-world problem (e.g., automating customer support).

---

### Assessment Plan

| Assessment Method      | Description                          | Weightage (%) |
|------------------------|--------------------------------------|---------------|
| Quizzes               | Multiple-choice quizzes after each module | 30%          |
| Assignments           | Practical exercises for each module  | 40%           |
| Final Project         | A comprehensive project combining all skills | 20%          |
| Participation         | Active participation in discussions  | 10%           |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve a minimum score of 70% across all assessments.
2. Submit all interactive assignments and the final project.
3. Actively participate in at least 80% of discussion forums and live sessions.

---

### Marketing Strategy

#### Step 1: Social Media Campaigns
- Create engaging posts highlighting the course benefits, modules, and outcomes.
- Share testimonials and success stories from beta testers or early participants.
- Use targeted ads on platforms like LinkedIn, Facebook, and Instagram.

#### Step 2: Email Marketing
- Build an email list of potential learners (e.g., professionals, students).
- Send a series of emails introducing the course, its benefits, and enrollment deadlines.
- Include exclusive early-bird discounts and referral incentives.

#### Step 3: Partnerships
- Collaborate with universities, coding bootcamps, and AI communities to promote the course.
- Partner with influencers and thought leaders in the AI space for endorsements.

#### Step 4: Content Creation
- Publish blog posts, videos, and webinars on topics related to prompt engineering.
- Share free resources (e.g., mini-guides, templates) to attract interest.
- Host live Q&A sessions to engage with the audience and answer questions about the course.

#### Step 5: Testimonials and Reviews
- Collect feedback from early participants and showcase their success stories.
- Encourage learners to share their experiences on social media and review platforms.

---

In [55]:
outline_prompt = """Create a course outline for a beginner online course. Use Markdown and follow this structure exactly.

Start with the line `## Course Outline: Mastering Prompt Engineering for AI Applications`, then a `---` line.

### Course Overview
- One paragraph (3-4 sentences): what the course teaches, why prompt engineering matters and who it is for.
- `#### Objectives` with exactly 3 bullet points.
- `#### Target Audience` with exactly 3 bullet points.

### Module Breakdown
Exactly 5 modules that go from the basics to a final project. Every module uses this format:
#### Module N: <Title>
- **Description:** one or two sentences about the module.
- **Key Learning Outcomes:**
  1. ...
  2. ...
  3. ...
- **Interactive Assignment:** one practical hands-on task for the end of the module.
Put a `---` line after every module.

### Assessment Plan
A Markdown table with the columns `Assessment Method | Description | Weightage (%)`, 4 rows, weights adding up to 100%.

### Certification Criteria
The sentence "To earn a certificate of completion, students must:" followed by a numbered list of 3 requirements
(minimum overall score, submitted assignments and final project, participation).

### Marketing Strategy
5 steps, each as `#### Step N: <Title>` followed by 2-3 bullet points with concrete actions for promoting the course.

Separate the main sections with `---`. Do not write anything before the title or after the last section.
The content may contain new ideas but must stay about prompt engineering for beginners."""

outline = ask(outline_prompt).choices[0].message.content
display(Markdown(outline))

## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course introduces the fundamentals of prompt engineering and shows learners how to communicate effectively with AI tools to produce clearer, more accurate, and more useful outputs. Students will learn how to structure prompts, refine responses, and apply best practices across common AI use cases such as writing, brainstorming, summarization, and problem-solving. Prompt engineering matters because the quality of an AI system’s output often depends on the quality of the input it receives. This course is designed for beginners, professionals, students, and anyone who wants to use AI more effectively in everyday tasks or work.

#### Objectives
- Understand the core principles of effective prompt design for AI applications.
- Learn how to improve AI responses through iteration, constraints, and context.
- Build confidence using prompt engineering techniques in practical scenarios.

#### Target Audience
- Beginners with little or no experience using AI tools.
- Professionals seeking to improve productivity with AI-assisted workflows.
- Students and creators interested in generating better content with AI.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module explains what prompt engineering is, how AI models interpret prompts, and why wording matters. Learners will explore simple examples of strong and weak prompts.
- **Key Learning Outcomes:**
  1. Define prompt engineering and its role in AI applications.
  2. Identify the elements of a clear and effective prompt.
  3. Recognize common beginner mistakes when prompting AI.
- **Interactive Assignment:** Rewrite five vague prompts into clear, specific prompts for different AI tasks.
---

#### Module 2: Writing Clear and Structured Prompts
- **Description:** Learners will study methods for organizing prompts with context, instructions, and expected output formats. This module focuses on making prompts easier for AI systems to follow.
- **Key Learning Outcomes:**
  1. Use context to guide AI responses more effectively.
  2. Create prompts with explicit instructions and desired formats.
  3. Apply role-based prompting for better task alignment.
- **Interactive Assignment:** Create three structured prompts: one for writing, one for analysis, and one for brainstorming.
---

#### Module 3: Improving Responses Through Iteration
- **Description:** This module teaches how to refine prompts after reviewing AI outputs. Learners will practice adjusting prompts to reduce ambiguity and improve relevance, accuracy, and consistency.
- **Key Learning Outcomes:**
  1. Evaluate AI outputs for quality and usefulness.
  2. Revise prompts based on response gaps or errors.
  3. Apply iteration strategies to achieve better results.
- **Interactive Assignment:** Test one prompt in multiple versions and compare the output quality from each iteration.
---

#### Module 4: Prompt Techniques for Common Use Cases
- **Description:** Learners will apply prompt engineering to practical tasks such as summarizing text, drafting emails, generating ideas, and solving simple problems. The module highlights how different goals require different prompting styles.
- **Key Learning Outcomes:**
  1. Tailor prompts for specific everyday and professional use cases.
  2. Use constraints to control tone, length, and format.
  3. Select prompt techniques that match the task objective.
- **Interactive Assignment:** Design a prompt toolkit with four prompts for four different real-world scenarios.
---

#### Module 5: Final Project — Designing a Prompt Workflow
- **Description:** In the final module, learners will build a small prompt-based workflow for a chosen use case. They will demonstrate how to move from a basic prompt to a refined, reusable prompt system.
- **Key Learning Outcomes:**
  1. Combine multiple prompt engineering techniques into one workflow.
  2. Develop prompts that can be reused and adapted.
  3. Present a practical prompt solution for a real scenario.
- **Interactive Assignment:** Create and present a final prompt workflow for a chosen task, including prompt drafts and refinements.
---

### Assessment Plan

| Assessment Method | Description | Weightage (%) |
|---|---|---:|
| Module Quizzes | Short quizzes after key lessons to check understanding of core concepts | 20 |
| Interactive Assignments | Practical tasks completed in each module to demonstrate prompt-writing skills | 30 |
| Participation and Discussion | Engagement in course discussions, feedback exchanges, and practice activities | 10 |
| Final Project | A complete prompt workflow project showing mastery of beginner techniques | 40 |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve a minimum overall score of 70%.
2. Submit all required module assignments and the final project.
3. Participate in course discussions or feedback activities at least once per module.

---

### Marketing Strategy

#### Step 1: Highlight Beginner-Friendly Benefits
- Emphasize that no technical background is required.
- Showcase simple outcomes like better writing, faster brainstorming, and improved AI results.
- Use before-and-after prompt examples in promotional materials.

#### Step 2: Create Short Demo Content
- Publish short videos or carousel posts showing prompt improvements in action.
- Offer a free sample lesson on writing effective prompts.
- Include quick tips that demonstrate immediate value.

#### Step 3: Use Targeted Audience Messaging
- Promote the course to students, professionals, and content creators separately.
- Tailor landing page copy to common use cases such as productivity, writing, and learning.
- Run ads or posts around practical AI skills for beginners.

#### Step 4: Build Trust with Social Proof
- Share testimonials from early learners or beta testers.
- Feature example outputs from course exercises and final projects.
- Highlight the course structure, assignments, and certification value.

#### Step 5: Offer Launch Promotions
- Provide early-bird discounts or bundle offers for the first cohort.
- Create a limited-time enrollment bonus, such as a prompt cheat sheet.
- Use email campaigns to remind prospects about deadlines and course benefits.

## Course Outline: Mastering Prompt Engineering for AI Applications

---

### Course Overview
This beginner-friendly course introduces the fundamentals of prompt engineering and shows learners how to communicate effectively with AI tools to produce clearer, more accurate, and more useful outputs. Students will learn how to structure prompts, refine responses, and apply best practices across common AI use cases such as writing, brainstorming, summarization, and problem-solving. Prompt engineering matters because the quality of an AI system’s output often depends on the quality of the input it receives. This course is designed for beginners, professionals, students, and anyone who wants to use AI more effectively in everyday tasks or work.

#### Objectives
- Understand the core principles of effective prompt design for AI applications.
- Learn how to improve AI responses through iteration, constraints, and context.
- Build confidence using prompt engineering techniques in practical scenarios.

#### Target Audience
- Beginners with little or no experience using AI tools.
- Professionals seeking to improve productivity with AI-assisted workflows.
- Students and creators interested in generating better content with AI.

---

### Module Breakdown

#### Module 1: Introduction to Prompt Engineering
- **Description:** This module explains what prompt engineering is, how AI models interpret prompts, and why wording matters. Learners will explore simple examples of strong and weak prompts.
- **Key Learning Outcomes:**
  1. Define prompt engineering and its role in AI applications.
  2. Identify the elements of a clear and effective prompt.
  3. Recognize common beginner mistakes when prompting AI.
- **Interactive Assignment:** Rewrite five vague prompts into clear, specific prompts for different AI tasks.
---

#### Module 2: Writing Clear and Structured Prompts
- **Description:** Learners will study methods for organizing prompts with context, instructions, and expected output formats. This module focuses on making prompts easier for AI systems to follow.
- **Key Learning Outcomes:**
  1. Use context to guide AI responses more effectively.
  2. Create prompts with explicit instructions and desired formats.
  3. Apply role-based prompting for better task alignment.
- **Interactive Assignment:** Create three structured prompts: one for writing, one for analysis, and one for brainstorming.
---

#### Module 3: Improving Responses Through Iteration
- **Description:** This module teaches how to refine prompts after reviewing AI outputs. Learners will practice adjusting prompts to reduce ambiguity and improve relevance, accuracy, and consistency.
- **Key Learning Outcomes:**
  1. Evaluate AI outputs for quality and usefulness.
  2. Revise prompts based on response gaps or errors.
  3. Apply iteration strategies to achieve better results.
- **Interactive Assignment:** Test one prompt in multiple versions and compare the output quality from each iteration.
---

#### Module 4: Prompt Techniques for Common Use Cases
- **Description:** Learners will apply prompt engineering to practical tasks such as summarizing text, drafting emails, generating ideas, and solving simple problems. The module highlights how different goals require different prompting styles.
- **Key Learning Outcomes:**
  1. Tailor prompts for specific everyday and professional use cases.
  2. Use constraints to control tone, length, and format.
  3. Select prompt techniques that match the task objective.
- **Interactive Assignment:** Design a prompt toolkit with four prompts for four different real-world scenarios.
---

#### Module 5: Final Project — Designing a Prompt Workflow
- **Description:** In the final module, learners will build a small prompt-based workflow for a chosen use case. They will demonstrate how to move from a basic prompt to a refined, reusable prompt system.
- **Key Learning Outcomes:**
  1. Combine multiple prompt engineering techniques into one workflow.
  2. Develop prompts that can be reused and adapted.
  3. Present a practical prompt solution for a real scenario.
- **Interactive Assignment:** Create and present a final prompt workflow for a chosen task, including prompt drafts and refinements.
---

### Assessment Plan

| Assessment Method | Description | Weightage (%) |
|---|---|---:|
| Module Quizzes | Short quizzes after key lessons to check understanding of core concepts | 20 |
| Interactive Assignments | Practical tasks completed in each module to demonstrate prompt-writing skills | 30 |
| Participation and Discussion | Engagement in course discussions, feedback exchanges, and practice activities | 10 |
| Final Project | A complete prompt workflow project showing mastery of beginner techniques | 40 |

---

### Certification Criteria
To earn a certificate of completion, students must:
1. Achieve a minimum overall score of 70%.
2. Submit all required module assignments and the final project.
3. Participate in course discussions or feedback activities at least once per module.

---

### Marketing Strategy

#### Step 1: Highlight Beginner-Friendly Benefits
- Emphasize that no technical background is required.
- Showcase simple outcomes like better writing, faster brainstorming, and improved AI results.
- Use before-and-after prompt examples in promotional materials.

#### Step 2: Create Short Demo Content
- Publish short videos or carousel posts showing prompt improvements in action.
- Offer a free sample lesson on writing effective prompts.
- Include quick tips that demonstrate immediate value.

#### Step 3: Use Targeted Audience Messaging
- Promote the course to students, professionals, and content creators separately.
- Tailor landing page copy to common use cases such as productivity, writing, and learning.
- Run ads or posts around practical AI skills for beginners.

#### Step 4: Build Trust with Social Proof
- Share testimonials from early learners or beta testers.
- Feature example outputs from course exercises and final projects.
- Highlight the course structure, assignments, and certification value.

#### Step 5: Offer Launch Promotions
- Provide early-bird discounts or bundle offers for the first cohort.
- Create a limited-time enrollment bonus, such as a prompt cheat sheet.
- Use email campaigns to remind prospects about deadlines and course benefits.

####